# Next-Day Stress Prediction Model

## Model Overview

This model predicts the user's **next-day self-reported stress level** on a 0–100 scale.

The model combines the user's current and historical self-reported stress with wearable-derived behavioral, sleep, and physiological information.

### Inputs

- Current self-reported stress
- Historical mean stress
- Daily steps
- Overall sleep score
- Deep sleep duration
- HRV(RMSSD)

### Output

- Predicted self-reported stress level for the **next calendar day (0–100)**

This model is intended for **stress forecasting** when the user's current self-reported stress is available.

In [51]:
#based on the fomer training experiments, linear regression with multi-element data is the best model
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [52]:
PROJECT_ROOT = Path.cwd().parent

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "daily_model_v1.csv"
)

MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "stress_predictor_multimodal_v1.joblib"
)

print("Data:", DATA_PATH)
print("Model:", MODEL_PATH)

Data: e:\schoolworks\y3s1\DIP\MindWatch-ML\data\processed\daily_model_v1.csv
Model: e:\schoolworks\y3s1\DIP\MindWatch-ML\models\stress_predictor_multimodal_v1.joblib


In [53]:
df = pd.read_csv(DATA_PATH)

df["date"] = pd.to_datetime(df["date"])

print("Dataset shape:", df.shape)
print("Participants:", df["participant_id"].nunique())
print("Date range:", df["date"].min(), "to", df["date"].max())

df.head()

Dataset shape: (3118, 14)
Participants: 35
Date range: 2025-02-14 00:00:00 to 2025-07-09 00:00:00


,participant_id,date,current_stress,anxiety,avg_rmssd,overall_score,deep_sleep_in_minutes,daily_steps,STRESS_SCORE,next_date,next_stress_observed,days_to_next,next_day_stress,participant_historical_mean
0,1,2025-02-19,15.0,0.0,NaN,NaN,NaN,NaN,NaN,2025-02-20,33.0,1.0,33.0,NaN
1,1,2025-02-20,33.0,0.0,NaN,NaN,NaN,NaN,NaN,2025-02-21,12.0,1.0,12.0,15.00
2,1,2025-02-21,12.0,0.0,NaN,NaN,NaN,3057.0,NaN,2025-02-22,9.0,1.0,9.0,24.00
3,1,2025-02-22,9.0,8.0,NaN,NaN,NaN,3168.0,NaN,2025-02-23,4.0,1.0,4.0,20.00
4,1,2025-02-23,4.0,7.0,69.259534,78.0,61.0,4287.0,58.0,2025-02-24,37.0,1.0,37.0,17.25


In [54]:
FINAL_FEATURES = [
    "current_stress",
    "participant_historical_mean",
    "daily_steps",
    "overall_score",
    "deep_sleep_in_minutes",
    "avg_rmssd"
]

TARGET = "next_day_stress"

model_df = (
    df[df[TARGET].notna()]
    .sort_values(["date", "participant_id"])
    .reset_index(drop=True)
)

print("Final features:")
for feature in FINAL_FEATURES:
    print("-", feature)

print("\nUsable prediction rows:", len(model_df))

print("\nMissing values:")
print(model_df[FINAL_FEATURES].isna().sum())

Final features:
- current_stress
- participant_historical_mean
- daily_steps
- overall_score
- deep_sleep_in_minutes
- avg_rmssd

Usable prediction rows: 2726

Missing values:
current_stress                    0
participant_historical_mean      30
daily_steps                     731
overall_score                  1379
deep_sleep_in_minutes          1379
avg_rmssd                      1269
dtype: int64


In [55]:
unique_dates = np.sort(model_df["date"].unique())

split_idx = int(len(unique_dates) * 0.8)
cutoff_date = unique_dates[split_idx]

train_df = model_df[
    model_df["date"] < cutoff_date
].copy()

test_df = model_df[
    model_df["date"] >= cutoff_date
].copy()

X_train = train_df[FINAL_FEATURES]
X_test = test_df[FINAL_FEATURES]

y_train = train_df[TARGET]
y_test = test_df[TARGET]

print("Cutoff date:", cutoff_date)
print("Training samples:", len(train_df))
print("Test samples:", len(test_df))

print(
    "Train period:",
    train_df["date"].min(),
    "to",
    train_df["date"].max()
)

print(
    "Test period:",
    test_df["date"].min(),
    "to",
    test_df["date"].max()
)

Cutoff date: 2025-06-07T00:00:00.000000
Training samples: 2349
Test samples: 377
Train period: 2025-02-14 00:00:00 to 2025-06-06 00:00:00
Test period: 2025-06-07 00:00:00 to 2025-07-05 00:00:00


In [56]:
final_model = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "model",
        LinearRegression()
    )
])

final_model.fit(
    X_train,
    y_train
)

y_pred = final_model.predict(X_test)

In [57]:
mae = mean_absolute_error(
    y_test,
    y_pred
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred
    )
)

r2 = r2_score(
    y_test,
    y_pred
)

print(f"MAE:  {mae:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"R²:   {r2:.3f}")

MAE:  15.483
RMSE: 21.498
R²:   0.264


In [58]:
#train models
X_all = model_df[FINAL_FEATURES]
y_all = model_df[TARGET]

deployment_model = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "model",
        LinearRegression()
    )
])

deployment_model.fit(
    X_all,
    y_all
)

print("Deployment model trained.")
print("Training samples:", len(X_all))

Deployment model trained.
Training samples: 2726


In [59]:
#save final multimodal model
MODEL_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

joblib.dump(
    deployment_model,
    MODEL_PATH
)

print("Model saved to:")
print(MODEL_PATH)

print("\nFile exists:", MODEL_PATH.exists())

Model saved to:
e:\schoolworks\y3s1\DIP\MindWatch-ML\models\stress_predictor_multimodal_v1.joblib

File exists: True
